# Arxiv Recommender GPU Workbench
대시보드에서 연결되는 GPU 학습·벤치마크 워크벤치입니다. **런타임 > 런타임 유형 변경 > GPU**를 선택한 뒤 위에서부터 실행하세요.

In [ ]:
import os, subprocess, sys, torch
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다. 런타임 유형을 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)

In [ ]:
REPOSITORY = 'https://github.com/YHFTF/arxiv-conversational-recommender.git' # @param {type:'string'}
BRANCH = 'main' # @param {type:'string'}
PROJECT_ROOT = '/content/arxiv-recsys'
if os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
    subprocess.run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_ROOT, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPOSITORY, PROJECT_ROOT], check=True)
os.chdir(PROJECT_ROOT)
# Colab가 제공하는 CUDA 대응 PyTorch를 그대로 사용한다.
# 누락된 프로젝트 보조 패키지만 설치하며 torch/CUDA는 pip로 변경하지 않는다.
import importlib.util
packages = {'torch_geometric': 'torch-geometric==2.6.1', 'networkx': 'networkx==3.4.2', 'numpy': 'numpy==1.26.4', 'pandas': 'pandas==2.2.3', 'ogb': 'ogb==1.3.6', 'sklearn': 'scikit-learn==1.6.1', 'openai': 'openai==1.61.1', 'dotenv': 'python-dotenv==1.0.1', 'requests': 'requests==2.32.3'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('누락 패키지 설치:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('프로젝트 보조 패키지가 이미 준비되어 있습니다.')
print('기존 PyTorch 유지:', torch.__version__, '| CUDA:', torch.version.cuda)

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')
DATA_ARCHIVE = '/content/drive/MyDrive/colab_data.zip' # @param {type:'string'}
SHARED_OUTPUT = '/content/drive/MyDrive/arxiv-recsys-output' # @param {type:'string'}
if os.path.exists(DATA_ARCHIVE):
    subprocess.run(['unzip', '-q', '-o', DATA_ARCHIVE, '-d', PROJECT_ROOT], check=True)
required = ['subdataset/build_hetero_graph_v2.pt', 'subdataset/arxiv_master_final.json', 'output/knowledge_meta.json']
missing = [path for path in required if not os.path.exists(os.path.join(PROJECT_ROOT, path))]
assert not missing, f'필수 데이터가 없습니다: {missing}. {DATA_ARCHIVE}를 확인하세요.'
os.makedirs(SHARED_OUTPUT, exist_ok=True)
project_output = os.path.join(PROJECT_ROOT, 'output')
if os.path.isdir(project_output): shutil.copytree(project_output, SHARED_OUTPUT, dirs_exist_ok=True)
print('데이터 준비 완료 · 공유 Output:', SHARED_OUTPUT)

In [ ]:
TASK = 'train_v4' # @param ['train_v4', 'benchmark_v2', 'inference_v4']
INFERENCE_QUERY = 'graph neural networks' # @param {type:'string'}
commands = {
    'train_v4': [sys.executable, 'code/model/train_v4_knowledge_bpr.py'],
    'benchmark_v2': [sys.executable, 'code/test/run_benchmark_v2.py'],
    'inference_v4': [sys.executable, 'code/model/inference_v4.py', INFERENCE_QUERY],
}
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['OUTPUT_DIR'] = SHARED_OUTPUT
print('실행:', ' '.join(commands[TASK]))
subprocess.run(commands[TASK], cwd=PROJECT_ROOT, env=env, check=True)

In [ ]:
print('결과는 모든 실행에서 같은 공유 폴더에 저장됩니다:', SHARED_OUTPUT)
print('저장 파일:', sorted(os.listdir(SHARED_OUTPUT))[:50])